In [2]:
# run for only first time

# !git clone https://github.com/thuanz123/realfill.git

In [2]:
# run for only first time
# %cd realfill
# %ls

In [1]:
# !pip install -r /content/realfill/requirements.txt
!pip install -r requirements.txt

  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


In [4]:
from accelerate.utils import write_basic_config
write_basic_config()|

/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration already exists at /userhome/cs/zc923404/.cache/huggingface/accelerate/default_config.yaml, will not override. Run `accelerate config` manually or pass a different `save_location`.


False

In [26]:
%pip install "numpy<2"
import numpy
numpy.__version__

  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


'1.26.4'

In [27]:
%pip install diffusers transformers accelerate scipy safetensors 

Note: you may need to restart the kernel to use updated packages.


In [28]:
import torch
import os
from diffusers import StableDiffusionInpaintPipeline

# 1. Define the official Model ID
model_id = "sd2-community/stable-diffusion-2-inpainting"

# 2. Check for device (NVIDIA CUDA for the farm, MPS for your Mac)
if torch.cuda.is_available():
    current_device = "cuda"
    current_dtype = torch.float16 # Farm GPUs handle float16 for speed
elif torch.backends.mps.is_available():
    current_device = "mps"
    current_dtype = torch.float32 # Mac is more stable with float32
else:
    current_device = "cpu"
    current_dtype = torch.float32

print(f"Loading pipeline onto: {current_device}")

# 3. Load the pipeline (It will download once and then cache automatically)
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    model_id,
    torch_dtype=current_dtype,
    use_safetensors=True
)

pipe = pipe.to(current_device)

Loading pipeline onto: cuda


/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading pipeline components...: 100%|█████████████| 6/6 [00:42<00:00,  7.13s/it]


In [ ]:
# from huggingface_hub import login

# base brain: stable diffusion 2.0 inpainting
%env MODEL_NAME=sd2-community/stable-diffusion-2-inpainting
# reference image
%env TRAIN_DIR=data/flowerwoman
%env OUTPUT_DIR=flowerwoman-model

# %env TRAIN_DIR=data/chiwah
# %env OUTPUT_DIR=chiwah-model

# !accelerate launch train_realfill.py \
!accelerate launch train_realfill.py \
  --pretrained_model_name_or_path=$MODEL_NAME \
  --train_data_dir=$TRAIN_DIR \
  --output_dir=$OUTPUT_DIR \
  --mixed_precision=fp16 \
  --resolution=512 \
  --train_batch_size=16 \
  --gradient_accumulation_steps=1 \
  --unet_learning_rate=2e-4 \
  --text_encoder_learning_rate=4e-5 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=100 \
  --max_train_steps=2000 \
  --lora_rank=8 \
  --lora_dropout=0.1 \
  --lora_alpha=16

  #   # --train_batch_size=16 \

env: MODEL_NAME=sd2-community/stable-diffusion-2-inpainting
env: TRAIN_DIR=data/flowerwoman
env: OUTPUT_DIR=flowerwoman-model
/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transfo

In [3]:
%env MODEL_PATH=flowerwoman-model
%env VAL_IMG=data/flowerwoman/target/target.png
%env VAL_MASK=data/flowerwoman/target/mask.png
%env OUTPUT_IMG_DIR=chiwah-results


!accelerate launch infer.py \
    --model_path=$MODEL_PATH \
    --validation_image=$VAL_IMG \
    --validation_mask=$VAL_MASK \
    --output_dir=$OUTPUT_IMG_DIR

env: MODEL_PATH=flowerwoman-model
env: VAL_IMG=data/flowerwoman/target/target.png
env: VAL_MASK=data/flowerwoman/target/mask.png
env: OUTPUT_IMG_DIR=chiwah-results

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/userhome/cs/zc923404/miniconda3/envs/apai3010/bin/accelerate", line 3, in <module>
    from accelerate.commands.accelerate_cli import main
  File "/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/accelerate/commands/accelerate_cli.py", line 21, in <module>
    from accelerate.commands.estimate import estimate_command_parser
  

In [4]:
!python -m pip install "numpy<2.0"

  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [1]:
import torchmetrics
import open_clip
import lpips
from skimage import metrics

print(f"Benchmark dependencies verified.")
# print(f"LPIPS version: {lpips.__version__}")
# Functional check for LPIPS
try:
    _ = lpips.LPIPS(net='vgg')
    print("LPIPS is installed and functional.")
except Exception as e:
    print(f"LPIPS check failed: {e}")

print(f"Benchmark dependencies verified.")

/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Benchmark dependencies verified.
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/lpips/weights/v0.1/vgg.pth
LPIPS is installed and functional.
Benchmark dependencies verified.


In [3]:
import benchmark_suite
import importlib
importlib.reload(benchmark_suite)
from benchmark_suite import RealFillBench

# Initialize evaluator
evaluator = RealFillBench(device="cuda")

# Test on one existing output
sample = "m_flower"
gen_path = f"./outputs/{sample}/generated.png"
target_path = f"./data/{sample}/target/target.png"

results = evaluator.calculate_all(gen_path, target_path)
print(f"Sanity Check Results for {sample}:", results)

ImportError: cannot import name 'PeakSignalToNoiseRatio' from 'torchmetrics' (/userhome/cs/zc923404/miniconda3/envs/apai3010/lib/python3.10/site-packages/torchmetrics/__init__.py)

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import importlib

import preprocess
importlib.reload(preprocess)

from PIL import Image
from benchmark_suite import RealFillBench
from preprocess import preprocess_sample 

evaluator = RealFillBench()
all_results = [] 

# Corrected paths based on your screenshots
base_data_path = "./realfill_data_release_full/Qualitative"
base_output_path = "./Experiments"
categories = ["Luminance_Test", "Geometry_Test", "Subject_Test", "Perspective_Test"]

USE_REFINED_LUMINANCE = True 

# Iterate logic
    # for category in categories:
    #     cat_path = os.path.join(base_data_path, category)
    #     if not os.path.exists(cat_path): continue
        
    #     # Get all sample folder names (01, 04, 23, etc.)
    #     samples = sorted([s for s in os.listdir(cat_path) if os.path.isdir(os.path.join(cat_path, s))])
        
    #     for sample in samples:
    #         sample_path = os.path.join(cat_path, sample)
    #         print(f"\n Processing {category} | Sample {sample}")
            
    #         # --- 1. PREPROCESS ---
    #         if USE_REFINED_LUMINANCE:
    #             preprocess_sample(sample_path)
    #             train_dir = os.path.join(sample_path, "input_refined")
    #         else:
    #             train_dir = os.path.join(sample_path, "input")
                
    #         # --- 2. DYNAMIC PATHS FOR TRAINING/INFER ---
    #         output_dir = os.path.abspath(os.path.join(base_output_path, category, sample, "model"))
    #         final_img_dir = os.path.abspath(os.path.join(base_output_path, category, sample, "results"))
    #         val_img = os.path.join(sample_path, "target", "target.png")
    #         val_mask = os.path.join(sample_path, "target", "mask.png")

        # --- 3. TRAIN & INFER ---
        !accelerate launch train_realfill.py --train_data_dir={train_dir} --output_dir={output_dir} --max_train_steps=2000 --lora_rank=8
        !accelerate launch infer.py --model_path={output_dir} --validation_image={val_img} --validation_mask={val_mask} --output_dir={final_img_dir}
            
        # --- 4. EVALUATE & STITCH ---
        generated_path = os.path.join(final_img_dir, "0.png")
        if os.path.exists(generated_path):
            scores = evaluator.calculate_all(generated_path, val_img, val_mask)
            all_results.append({"Category": category, "Sample": sample, **scores})
            
            # Create Paper-Style Strip
            try:
                # Grab the first reference image from the input folder
                ref_list = sorted(glob.glob(os.path.join(sample_path, "input", "*.png")))
                ref_img = Image.open(ref_list[0]).resize((512, 512))
                tgt_img = Image.open(val_img).resize((512, 512))
                res_img = Image.open(generated_path).resize((512, 512))
                
                fig, axes = plt.subplots(1, 3, figsize=(12, 4))
                axes[0].imshow(ref_img); axes[0].set_title("Reference")
                axes[1].imshow(tgt_img); axes[1].set_title("Target (GT)")
                axes[2].imshow(res_img); axes[2].set_title(f"RealFill Result")
                for ax in axes: ax.axis('off')
                plt.show()
            except Exception as e:
                print(f"Qualitative skip: {e}")

# --- 5. TIDY OUTPUT ---
df = pd.DataFrame(all_results)
display(df.groupby('Category').mean(numeric_only=True).round(4))

NameError: name 'PeakSignalToNoiseRatio' is not defined

In [ ]:
import os
from benchmark_suite import RealFillBench

# Initialize the benchmark tool
evaluator = RealFillBench()

# Core Paths
base_data_path = "./realfill_data_release_full/Qualitative"
base_output_path = "./Experiments"
categories = ["Luminance_Test", "Geometry_Test", "Subject_Test", "Perspective_Test"]

# Toggle this to True to run your "Novel Improvement"
USE_REFINED_LUMINANCE = True

# The Loop
for category in categories:
    cat_path = os.path.join(base_data_path, category)
    if not os.path.exists(cat_path): continue
    
    for sample in os.listdir(cat_path):
        sample_path = os.path.join(cat_path, sample)
        if not os.path.isdir(sample_path): continue
            
        print(f"\n Processing: {category} / Sample {sample}")
        
        # --- 0. Dynamic Paths ---
        # Where the reference images are
        train_dir = os.path.abspath(sample_path)
        # Where the LoRA weights will be saved
        output_dir = os.path.abspath(os.path.join(base_output_path, category, sample, "model"))
        # Where the final images will be saved
        final_img_dir = os.path.abspath(os.path.join(base_output_path, category, sample, "results"))
        
        # Ground Truth paths for validation
        val_img = os.path.join(sample_path, "target", "target.png")
        val_mask = os.path.join(sample_path, "target", "mask.png")

        # --- 1. PREPROCESS (The Improvement Step) ---
        if USE_REFINED_LUMINANCE:
            print(f" Refining luminance for {sample}...")
            preprocess_sample(sample_path)
            # Point training to the refined folder
            train_dir = os.path.join(sample_path, "input_refined")
        else:
            train_dir = os.path.join(sample_path, "input")
            
        # --- 2. RUN TRAINING ---
        # We pass the python variables directly into the shell command using {var}
        !accelerate launch train_realfill.py \
          --pretrained_model_name_or_path="sd2-community/stable-diffusion-2-inpainting" \
          --train_data_dir={train_dir} \
          --output_dir={output_dir} \
          --mixed_precision=fp16 \
          --resolution=512 \
          --train_batch_size=1 \
          --max_train_steps=2000 \
          --lora_rank=8
          
        # --- 3. RUN INFERENCE ---
        !accelerate launch infer.py \
            --model_path={output_dir} \
            --validation_image={val_img} \
            --validation_mask={val_mask} \
            --output_dir={final_img_dir}
            
        # --- 4. EVALUATION & STITCHING ---
        generated_path = os.path.join(final_img_dir, "0.png")
        val_img = os.path.join(sample_path, "target", "target.png")
        
        if os.path.exists(generated_path):
            # A. Get Benchmark Values
            scores = evaluator.calculate_all(generated_path, val_img, val_mask)
            
            # B. Store for the Table
            result_entry = {
                "Category": category,
                "Sample": sample,
                "Improvement": "Luminance-Refined" if USE_REFINED_LUMINANCE else "Baseline",
                **scores
            }
            all_results.append(result_entry)
            
            # C. Create Qualitative Paper Strip
            try:
                # Load images for the strip
                ref_img = Image.open(os.path.join(sample_path, "input", "0.png")).resize((256, 256))
                tgt_img = Image.open(val_img).resize((256, 256))
                res_img = Image.open(generated_path).resize((256, 256))
                
                fig, axes = plt.subplots(1, 3, figsize=(15, 5))
                axes[0].imshow(ref_img); axes[0].set_title("Reference")
                axes[1].imshow(tgt_img); axes[1].set_title("Target (Masked)")
                axes[2].imshow(res_img); axes[2].set_title(f"Result ({category})")
                for ax in axes: ax.axis('off')
                
                plt.tight_layout()
                strip_save_path = os.path.join(final_img_dir, "paper_comparison_strip.png")
                plt.savefig(strip_save_path)
                plt.show() # Display in Jupyter
            except Exception as e:
                print(f"Could not stitch strip for {sample}: {e}")

# --- 5. SHOW TIDY TABLE ---
df = pd.DataFrame(all_results)
print("\n" + "="*50)
print("AVERAGE PERFORMANCE PER CATEGORY")
print("="*50)
display(df.groupby(['Category', 'Improvement']).mean(numeric_only=True).round(4))

In [ ]:
# test RealFillBench 

import benchmark_suite
import importlib
importlib.reload(benchmark_suite)
from benchmark_suite import RealFillBench

# Initialize evaluator
evaluator = RealFillBench(device="cuda")

# Test on one existing output
sample = "m_flower"
gen_path = f"./outputs/{sample}/generated.png"
target_path = f"./data/{sample}/target/target.png"

results = evaluator.calculate_all(gen_path, target_path)
print(f"Sanity Check Results for {sample}:", results)